# Information Extraction with COPRO

This notebook applies DSPy's `COPRO` optimizer to a facility-support information extraction task.

The program reads each support message and predicts:

1. **Urgency**: `low`, `medium`, or `high`
2. **Sentiment**: `positive`, `neutral`, or `negative`
3. **Categories**: the facility-management categories mentioned in the message

The notebook covers:
- configuring the DSPy language model
- downloading and splitting the facility-support dataset
- defining a three-part DSPy extraction program
- evaluating the baseline program
- compiling the program with `COPRO`
- inspecting the optimized predictor instructions and reevaluating the result

The markdown in this notebook now matches the actual `COPRO` configuration used in the code cells.


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



## Configure Language Model

The language model comes from `LLM_PROVIDER` in `.env`. `supportvectors` uses the SV cluster (`gpt-oss-20b`); any other value uses hosted OpenAI (`gpt-5-mini`).


In [2]:
# LLM_PROVIDER in .env selects the backend:
#   supportvectors -> SV cluster (gpt-oss-20b)
#   anything else  -> hosted OpenAI (gpt-5-mini; needs OPENAI_API_KEY)
import dspy
from svlearn_dspy.llm import make_lm

lm = make_lm()
dspy.configure(lm=lm)


## Load and Prepare Dataset

Download the facility support analyzer dataset from the `llama-prompt-ops` repository and convert each entry into a `dspy.Example` for DSPy.


In [3]:
import requests
import dspy
import json
import random

def init_dataset():
    # Load from the url
    url = "https://raw.githubusercontent.com/meta-llama/llama-prompt-ops/refs/heads/main/use-cases/facility-support-analyzer/dataset.json"
    dataset = json.loads(requests.get(url).text)
    dspy_dataset = [
        dspy.Example({
            "message": d['fields']['input'],
            "answer": d['answer'],
        }).with_inputs("message")
        for d in dataset
    ]
    random.Random(0).shuffle(dspy_dataset)
    train_set = dspy_dataset[:int(len(dspy_dataset) * 0.33)]
    val_set = dspy_dataset[int(len(dspy_dataset) * 0.33):int(len(dspy_dataset) * 0.66)]
    test_set = dspy_dataset[int(len(dspy_dataset) * 0.66):]

    return train_set, val_set, test_set

## Initialize Dataset Splits

Shuffle the dataset with a fixed seed and split it into training, validation, and test subsets. The output confirms the size of each split.


In [4]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(66, 66, 68)

## Examine Example Data

Inspect a sample message and its gold answer to understand the structure that the program needs to extract.


In [5]:
print("Input Message:")
print(train_set[0]['message'])

print("\n\nGold Answer:")
for k, v in json.loads(train_set[0]['answer']).items():
    print(f"{k}: {v}")

Input Message:
Subject: Adjusting Bi-Weekly Cleaning Schedule for My Office

Dear ProCare Facility Solutions Support Team,

I hope this message finds you well. My name is Dr. Alex Turner, and I have been utilizing your services for my office space for the past year. I must say, your team's dedication to maintaining a pristine environment has been commendable and greatly appreciated.

I am reaching out to discuss the scheduling of our regular cleaning services. While I find the logistical challenges of coordinating these services intellectually stimulating, I believe we could optimize the current schedule to better suit the needs of my team and our workflow. Specifically, I would like to explore the possibility of adjusting our cleaning schedule to a bi-weekly arrangement, ideally on Tuesdays and Fridays, to ensure our workspace remains consistently clean without disrupting our research activities.

Previously, I have attempted to adjust the schedule through the online portal, but I enc

## Define DSPy Module

Define the information-extraction program:
- one signature for urgency
- one signature for sentiment
- one signature for categories
- a module that runs all three `ChainOfThought` predictors and combines their outputs


In [6]:
from typing import List, Literal


class FacilitySupportAnalyzerUrgency(dspy.Signature):
    """
    Read the provided message and determine the urgency.
    """
    message: str = dspy.InputField()
    urgency: Literal['low', 'medium', 'high'] = dspy.OutputField()

class FacilitySupportAnalyzerSentiment(dspy.Signature):
    """
    Read the provided message and determine the sentiment.
    """
    message: str = dspy.InputField()
    sentiment: Literal['positive', 'neutral', 'negative'] = dspy.OutputField()

class FacilitySupportAnalyzerCategories(dspy.Signature):
    """
    Read the provided message and determine the set of categories applicable to the message.
    """
    message: str = dspy.InputField()
    categories: List[Literal["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmental_practices", "training_and_support_requests", "cleaning_services_scheduling", "customer_feedback_and_complaints", "facility_management_issues"]] = dspy.OutputField()

class FacilitySupportAnalyzerMM(dspy.Module):
    def __init__(self):
        self.urgency_module = dspy.ChainOfThought(FacilitySupportAnalyzerUrgency)
        self.sentiment_module = dspy.ChainOfThought(FacilitySupportAnalyzerSentiment)
        self.categories_module = dspy.ChainOfThought(FacilitySupportAnalyzerCategories)
    
    def forward(self, message: str):
        urgency = self.urgency_module(message=message)
        sentiment = self.sentiment_module(message=message)
        categories = self.categories_module(message=message)

        return dspy.Prediction(
            urgency=urgency.urgency,
            sentiment=sentiment.sentiment,
            categories=categories.categories
        )

program = FacilitySupportAnalyzerMM()

## Define Evaluation Metrics

Implement scoring functions for the three outputs and combine them into one metric. This metric is used for both baseline testing and `COPRO` optimization.


In [7]:
def score_urgency(gold_urgency, pred_urgency):
    """
    Compute score for the urgency module.
    """
    score = 1.0 if gold_urgency == pred_urgency else 0.0
    return score

def score_sentiment(gold_sentiment, pred_sentiment):
    """
    Compute score for the sentiment module.
    """
    score = 1.0 if gold_sentiment == pred_sentiment else 0.0
    return score

def score_categories(gold_categories, pred_categories):
    """
    Compute score for the categories module.
    Uses the same match/mismatch logic as category accuracy in the score.
    """
    correct = 0
    for k, v in gold_categories.items():
        if v and k in pred_categories:
            correct += 1
        elif not v and k not in pred_categories:
            correct += 1
    score = correct / len(gold_categories)
    return score

def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard for categories, sentiment, and urgency.
    Returns the score (float).
    """
    # Parse gold standard from example
    gold = json.loads(example['answer'])

    # Compute scores for all modules
    score_urgency_val = score_urgency(gold['urgency'], pred.urgency)
    score_sentiment_val = score_sentiment(gold['sentiment'], pred.sentiment)
    score_categories_val = score_categories(gold['categories'], pred.categories)

    # Overall score: average of the three accuracies
    total = (score_urgency_val + score_sentiment_val + score_categories_val) / 3

    return total

## Evaluate Baseline Performance

Measure the initial performance of the unoptimized program on the test set before running `COPRO`.


In [8]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 52.47 / 68 (77.2%): 100%|██████████| 68/68 [00:00<00:00, 68.17it/s]

2026/10/20 17:28:04 INFO dspy.evaluate.evaluate: Average Metric: 52.46666666666667 / 68 (77.2%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, tr...",✔️ [0.933]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, quality_and_safety_concerns, facili...",✔️ [0.933]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, facility_management_issues]",✔️ [0.633]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.633]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, cu...",✔️ [0.600]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[facility_management_issues, quality_and_safety_concerns]",✔️ [0.967]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[training_and_support_requests, general_inquiries]",✔️ [0.967]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,negative,"[routine_maintenance_requests, facility_management_issues, custome...",✔️ [0.667]


EvaluationResult(score=77.16, results=<list of 68 results>)

## Initialize COPRO Optimizer

Create the `COPRO` optimizer using the configured language model as `prompt_model`, along with the notebook's metric and search settings such as `breadth`, `depth`, and `init_temperature`.


In [9]:
from dspy import COPRO

optimizer = COPRO(
    prompt_model=lm,
    metric=metric,
    # Already a cheap search: breadth x depth = 3 x 2 candidate rounds.
    breadth=3,
    depth=2,
    # gpt-5 family models only accept the default temperature (1.0), so the
    # instruction-proposal temperature is pinned to 1.0 (COPRO's default 1.4
    # or a low 0.2 would 400 on gpt-5-mini).
    init_temperature=1.0,
)

## Run COPRO Optimization

Compile the program with `COPRO` using the training split and the evaluation settings passed through `eval_kwargs`. This produces an optimized version of the original prompt-based program.


In [10]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    eval_kwargs={
        "num_threads": 16,
        "display_progress": True,
        "display_table": False,
    },
)

2026/10/20 17:29:07 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/2.
2026/10/20 17:29:07 INFO dspy.teleprompt.copro_optimizer: At Depth 1/2, Evaluating Prompt Candidate #1/3 for Predictor 1 of 3.






[2026-10-20T17:29:07.310197]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2026/10/20 17:30:35 INFO dspy.evaluate.evaluate: Average Metric: 48.86666666666667 / 66 (74.0%)
2026/10/20 17:30:35 INFO dspy.teleprompt.copro_optimizer: At Depth 1/2, Evaluating Prompt Candidate #2/3 for Predictor 1 of 3.







[2026-10-20T17:30:35.799153]

System message:

Your input fields are:
1. `message` (str):
Your output fields are:
1. `reasoning` (str): 
2. `categories` (list[Literal['emergency_repair_services', 'routine_maintenance_requests', 'quality_and_safety_concerns', 'specialized_cleaning_services', 'general_inquiries', 'sustainability_and_environmental_practices', 'training_and_support_requests', 'cleaning_services_scheduling', 'customer_feedback_and_complaints', 'facility_management_issues']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## message ## ]]
{message}

[[ ## reasoning ## ]]
{reasoning}

[[ ## categories ## ]]
{categories}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmenta

2026/10/20 17:31:38 INFO dspy.evaluate.evaluate: Average Metric: 47.86666666666667 / 66 (72.5%)
2026/10/20 17:31:38 INFO dspy.teleprompt.copro_optimizer: At Depth 1/2, Evaluating Prompt Candidate #3/3 for Predictor 1 of 3.







[2026-10-20T17:31:38.801576]

System message:

Your input fields are:
1. `message` (str):
Your output fields are:
1. `reasoning` (str): 
2. `categories` (list[Literal['emergency_repair_services', 'routine_maintenance_requests', 'quality_and_safety_concerns', 'specialized_cleaning_services', 'general_inquiries', 'sustainability_and_environmental_practices', 'training_and_support_requests', 'cleaning_services_scheduling', 'customer_feedback_and_complaints', 'facility_management_issues']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## message ## ]]
{message}

[[ ## reasoning ## ]]
{reasoning}

[[ ## categories ## ]]
{categories}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmenta

... (48 outputs left out)

2026/10/20 17:46:42 INFO dspy.evaluate.evaluate: Average Metric: 50.13333333333333 / 66 (76.0%)
2026/10/20 17:46:42 INFO dspy.teleprompt.copro_optimizer: At Depth 2/2, Evaluating Prompt Candidate #6/6 for Predictor 3 of 3.







[2026-10-20T17:46:42.356538]

System message:

Your input fields are:
1. `message` (str):
Your output fields are:
1. `reasoning` (str): 
2. `categories` (list[Literal['emergency_repair_services', 'routine_maintenance_requests', 'quality_and_safety_concerns', 'specialized_cleaning_services', 'general_inquiries', 'sustainability_and_environmental_practices', 'training_and_support_requests', 'cleaning_services_scheduling', 'customer_feedback_and_complaints', 'facility_management_issues']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## message ## ]]
{message}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "categories": "{categories}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"emergency_repair_services\", \"routine_maintenance_requests\", \"quality

2026/10/20 17:48:34 INFO dspy.evaluate.evaluate: Average Metric: 50.93333333333333 / 66 (77.2%)







[2026-10-20T17:48:34.493841]

System message:

Your input fields are:
1. `message` (str):
Your output fields are:
1. `reasoning` (str): 
2. `categories` (list[Literal['emergency_repair_services', 'routine_maintenance_requests', 'quality_and_safety_concerns', 'specialized_cleaning_services', 'general_inquiries', 'sustainability_and_environmental_practices', 'training_and_support_requests', 'cleaning_services_scheduling', 'customer_feedback_and_complaints', 'facility_management_issues']]):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## message ## ]]
{message}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "categories": "{categories}        # note: the value you produce must adhere to the JSON schema: {\"type\": \"array\", \"items\": {\"type\": \"string\", \"enum\": [\"emergency_repair_services\", \"routine_maintenance_requests\", \"quality

## Inspect Optimized Prompts

Display the instructions for each predictor after `COPRO` finishes so we can see the prompts selected by the optimizer.


In [11]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: urgency_module.predict
Prompt:
Read the provided message and determine the urgency.
*********************************
Predictor: sentiment_module.predict
Prompt:
You will be given a single user message (a sentence or short paragraph). Determine its overall sentiment and output exactly one label — nothing else — chosen from: positive, negative, neutral, mixed.

Label definitions and decision rules:
- positive — clearly favorable, approving, or enthusiastic (praise, happiness, positive emojis).
- negative — clearly unfavorable, complaining, angry, or critical (complaints, negative emojis).
- neutral — factual, informational, or emotionally flat; or the sentiment is ambiguous/uncertain with no clear valence.
- mixed — the message contains clear positive and clear negative sentiments about the same topic or different aspects (e.g., praise for X but complaint about Y).

How to handle tricky cases:
- Sarcasm/irony: infer the intended sentiment (e.g., sarcastic “great” that actuall

## Evaluate Optimized Program

Run the optimized program on the test set to compare its final performance against the baseline.


In [12]:
evaluate(optimized_program)

Average Metric: 54.23 / 68 (79.8%): 100%|██████████| 68/68 [01:18<00:00,  1.16s/it]

2026/10/20 17:49:53 INFO dspy.evaluate.evaluate: Average Metric: 54.233333333333334 / 68 (79.8%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,[sustainability_and_environmental_practices],✔️ [1.000]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[quality_and_safety_concerns, routine_maintenance_requests]",✔️ [0.967]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,[routine_maintenance_requests],✔️ [0.667]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,[sustainability_and_environmental_practices],✔️ [1.000]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[general_inquiries, sustainability_and_environmental_practices]",✔️ [0.633]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,neutral,[sustainability_and_environmental_practices],✔️ [0.933]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[facility_management_issues, quality_and_safety_concerns]",✔️ [0.967]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,[training_and_support_requests],✔️ [1.000]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,negative,"[customer_feedback_and_complaints, facility_management_issues, qua...",✔️ [0.667]


EvaluationResult(score=79.75, results=<list of 68 results>)